# Non-Waste Images Extraction

We have satellite images in one folder (in .jpg format).
We have labels (yolo coordinates of bounding boxes) for each image in another folder (in .txt format).
Each bounding box marks a waste dump.

From each satellite image, this program extracts random sub-images that do not overlap with any of the bounding boxes of that image.
This creates our non-waste or class 0 dataset.

In [1]:
import cv2
import random
import os

In [3]:
# Paths to respective directories

src_folder_path = 'Waste_Images_All'
labels_folder_path = 'Waste_Labels_All'
class_0_folder_path = 'not_waste_images_all'

In [4]:
# Define size of extracted images
window_size = 150

In [ ]:
imgCount = 0
for filename in os.listdir(src_folder_path):

    # skip file if not in correct format
    if not filename.endswith(('.jpg', '.jpeg')):
        continue

    img_path = os.path.join(src_folder_path, filename)
    label_path = os.path.join(labels_folder_path, (filename[ : filename.rfind('.')] + '.txt'))

    # read image
    img = cv2.imread(img_path)

    # image dimensions
    img_height, img_width, _ = img.shape

     # list of bounding box rectangles
    bounding_boxes = []

    # append bounding boxes from labels file into list
    with open(label_path, 'r') as f:
        for line in f:
            # Extract the values from the line
            values = line.strip().split()
            x_centre, y_centre, w_norm, h_norm = map(float, values[1:])

            # Convert the normalized coordinates to pixel coordinates
            w = int(w_norm * img_width)
            h = int(h_norm * img_height)
            bbox_left = int((x_centre * img_width) - w/2)
            bbox_bottom = int((y_centre * img_height) + h/2)
            bbox_right = int((x_centre * img_width) + w/2)
            bbox_top = int((y_centre * img_height) - h/2)

            bounding_boxes.append((bbox_left, bbox_top, bbox_right, bbox_bottom))

    # Get 3 random windows (that do not overlap)
    windowCount = 0
    while not windowCount == 3:
        window_top = random.randint(0, img_height-window_size)
        window_left = random.randint(0, img_width-window_size)
        window_bottom = window_top + window_size
        window_right = window_left + window_size

        # Check whether window overlaps with bounding box
        overlaps = False
        for box in bounding_boxes:
            if window_right > box[0] and window_bottom > box[1] and window_left < box[2] and window_top < box[3]:
                overlaps = True
                break

        # save window as non-waste image if it does not overlap
        if not overlaps:
            imgCount += 1
            windowCount += 1
            subimage = img[window_top:window_bottom, window_left:window_right]
            cv2.imwrite(os.path.join(class_0_folder_path, (str(imgCount) + '.jpg')), subimage)

print("Extracted", imgCount, "Images")